# MEN 00 — Motion energy: a description against the task

Motion energy (ME) from the bottom and side cameras, from the aligned ME table
(`fip_motion_energy_aligned`, corrected Harp time, per-exposure ME on an even grid), set against
go cues, choices, outcomes and licks in the FIP sessions. No photometry is used.

0. Data, and a clock check: lick-triggered ME per session
1. Trial-averaged ME, aligned to the go cue and to the choice
2. Rewarded vs unrewarded trials
3. Responded vs no-response trials
4. Lick bouts that start within 2 s of a go cue (instructed) and later in the inter-trial
   interval (uninstructed)
5. ME events with and without licking
6. ME across the session (running average), with lick rate
7. ME by recent reward history (consecutive rewards and failures)
8. Numbers

Every average is taken within session, then within animal; plots show mean ± SEM across animals.
ME is normalised per session and camera (`ME_NORM`, z-score by default) before pooling, since its
scale depends on lighting, focus and framing.

## Setup

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import fip_utils as fu
import men_utils as mu
import signal_utils as su
import stats_utils as st
import behavior_utils as bu
import plot_utils as pu
from plotstyle import apply_style, style_ax, save_fig, PALETTE, OKABE_ITO

apply_style()
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="Degrees of freedom <= 0")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

In [ ]:
IS_CO = os.path.isdir("/root/capsule")

# Trial and event tables: the CSV-curated FIP assets (fip_05's candidates). The 4-channel rebuild
# must be attached on Code Ocean as DANE_4channels_curated for its animals to be included.
ASSET_CANDIDATES = {
    "3ch": ["/root/capsule/data/DANE_3channels_curated",
            "../../data/results-a278f830-9bf2-48e8-9928-a2ff78e4818f"],
    "4ch": ["/root/capsule/data/DANE_4channels_curated",
            "../../data/results-ddcccb0f-f18f-44a6-a2b1-caba680d28a1"],
}
# Folder holding fip_motion_energy_aligned/ (the ME table asset). Locally, point ME_DATA_ROOT at a
# folder with a copy of (part of) the table.
ME_DATA_ROOT = "/root/capsule/data" if IS_CO else os.environ.get("ME_DATA_ROOT", "../../data")
CACHE_PATH = ("/root/capsule/scratch/men_00/sessions.pkl" if IS_CO
              else os.environ.get("MEN_CACHE", "../../data/men_00_cache/sessions.pkl"))
FIG_DIR = "/root/capsule/scratch/figures/men" if IS_CO else "../data/figures/men"
SAVE_FIG = True

# ── Data ─────────────────────────────────────────────────────────────────────
CAMERAS = ("BottomCamera", "SideCameraRight")
EXCLUDE_ACTIONS = ()      # e.g. ("re-index",) to leave out the frame-drop sessions
FS = 100.0                # Hz; ME is bin-averaged from 500 Hz onto this grid
ME_NORM = "z"             # per session and camera: "z", "median" (multiples of the median) or "none"

# ── Windows (s) ──────────────────────────────────────────────────────────────
CUE_LAGS = np.arange(-2.0, 5.0 + 1e-9, 1 / FS)     # around the go cue; censored at the next cue
CHOICE_LAGS = np.arange(-1.0, 4.0 + 1e-9, 1 / FS)  # around the choice (first lick after the cue)
BOUT_LAGS = np.arange(-1.0, 3.0 + 1e-9, 1 / FS)    # around a lick-bout onset
EVENT_LAGS = np.arange(-1.0, 2.0 + 1e-9, 1 / FS)   # around an ME event onset
BASELINE_WIN = (-1.0, 0.0)    # before the go cue
CUE_WIN = (0.0, 1.0)          # after the go cue
OUTCOME_WIN = (0.5, 2.0)      # after the choice; consumption licking on rewarded trials

# ── Lick bouts (grouped by the library: a gap over 0.7 s starts a new bout) ────
INSTRUCTED_S = 2.0        # bouts starting within this many s after a go cue are "instructed"

# ── ME events ────────────────────────────────────────────────────────────────
ME_ONSET_KW = fu.ME_ONSET_KW   # 2.5 SD for 30 ms, one per 0.5 s: the fip_* movement-onset rule
LICK_WIN = (-0.5, 1.0)    # an ME event "has licking" if a lick falls in [onset-0.5, onset+1.0] s
N_SHIFT = 200             # circular shifts of the lick train for the chance level

# ── Session time course ──────────────────────────────────────────────────────
ROLL_WIN_S = 60.0
ROLL_STEP_S = 10.0
MIN_SESSIONS_AT_T = 3     # per animal: show a time point only where this many sessions reach it

# ── Colors ───────────────────────────────────────────────────────────────────
CAM_LABEL = {"BottomCamera": "Bottom camera", "SideCameraRight": "Side camera"}
C_ALL = PALETTE["all"]
C_REW, C_UNREW = OKABE_ITO["orange"], "#9b9b9b"
C_RESP, C_IGN, C_IGN_ISO = PALETTE["all"], OKABE_ITO["sky_blue"], OKABE_ITO["blue"]
C_INSTR, C_UNINSTR = OKABE_ITO["vermillion"], OKABE_ITO["blue"]
C_LICK, C_NOLICK = OKABE_ITO["bluish_green"], OKABE_ITO["reddish_purple"]

## 0. Data

`mu.inventory_sessions` lists every session in the ME table and finds its trial and event tables
in the FIP assets. A camera the ME table refused (Harp clock steps, trigger-log mismatches) has no
ME and is left out for that session; `EXCLUDE_ACTIONS` can leave out more by timing action.

In [ ]:
asset_roots = {}
for name, cands in ASSET_CANDIDATES.items():
    root = fu.find_asset_root(cands)
    print(f"{name}: {root}")
    if root:
        asset_roots[name] = root
assert asset_roots, "no FIP asset found; check ASSET_CANDIDATES"

me_index = fu.load_me_index(ME_DATA_ROOT)
inv = mu.inventory_sessions(asset_roots, me_index)
for cam in CAMERAS:
    inv[f"{cam}_used"] = (inv[f"{cam}_status"] == "ok") & ~inv[f"{cam}_action"].isin(EXCLUDE_ACTIONS)
inv["has_trials"] = inv["path"].notna()

print(f"{len(inv)} sessions in the ME table; {inv.has_trials.sum()} with trial tables")
missing = inv.loc[~inv.has_trials, "ses_idx"].tolist()
if missing:
    print(f"no trial table for {len(missing)} sessions (asset not attached?): {missing[:6]}{' ...' if len(missing) > 6 else ''}")
coverage = inv[inv.has_trials].groupby("subject").agg(
    sessions=("ses_idx", "size"),
    bottom=("BottomCamera_used", "sum"),
    side=("SideCameraRight_used", "sum"),
    asset=("asset", "first"),
)
coverage

In [ ]:
sessions = mu.load_sessions(inv[inv.has_trials], fs=FS, cache_path=CACHE_PATH,
                            data_root=ME_DATA_ROOT, cameras=CAMERAS)

# Normalise per session and camera; drop cameras excluded above
used = {(r.ses_idx, cam): bool(getattr(r, f"{cam}_used")) for r in inv.itertuples() for cam in CAMERAS}
for s in sessions:
    s["me_n"] = {cam: (mu.normalise(x, ME_NORM) if x is not None and used[(s["session"], cam)] else None)
                 for cam, x in s["me"].items()}
    tr = s["trials"]
    tr["responded"] = tr["animal_response"] != 2
    tr["rewarded"] = tr["earned_reward"].fillna(0) > 0
    tr["auto_water"] = tr["extra_reward"].fillna(0) > 0
    tr["next_cue"] = tr["goCue_start_time_in_session"].shift(-1)  # for censoring cue windows
    # Reward history entering this trial: num_reward_past of the previous trial
    tr["streak_before"] = tr["num_reward_past"].shift(1)

SUBJECTS = sorted({s["subject"] for s in sessions})
dur = np.array([len(next(x for x in s["me"].values() if x is not None)) / FS / 60
                for s in sessions if any(x is not None for x in s["me"].values())])
print(f"{len(sessions)} sessions from {len(SUBJECTS)} animals; "
      f"task period median {np.median(dur):.0f} min (range {dur.min():.0f}–{dur.max():.0f})")
for cam in CAMERAS:
    n = sum(s["me_n"][cam] is not None for s in sessions)
    print(f"  {CAM_LABEL[cam]}: {n} sessions")

### Clock check: lick-triggered ME

Each lick is a tongue protrusion that both cameras see, so ME averaged around lick times should
peak close to the lick. A session whose peak sits far from zero has a misplaced ME clock. The
lickometer registers contact, so a peak a few tens of ms *before* the lick is expected.

In [ ]:
LICK_LAGS = np.arange(-1.0, 1.0 + 1e-9, 1 / FS)
rows = []
lick_records = []
for s in sessions:
    for cam in CAMERAS:
        x = s["me_n"][cam]
        if x is None:
            continue
        seg = su.peri_event_grid(x, s["t0"], FS, s["licks"], LICK_LAGS)
        tr = np.nanmean(seg, axis=0)
        rows.append(dict(subject=s["subject"], session=s["session"], camera=cam,
                         n_licks=len(s["licks"]), peak_lag_s=LICK_LAGS[np.nanargmax(tr)],
                         peak=np.nanmax(tr)))
        lick_records.append(dict(subject=s["subject"], session=s["session"], camera=cam, trace=tr))
clock = pd.DataFrame(rows)
lick_records = pd.DataFrame(lick_records)
off = clock[clock.peak_lag_s.abs() > 0.15]
print("peak lag (s) by camera:")
print(clock.groupby("camera")["peak_lag_s"].describe()[["count", "mean", "min", "50%", "max"]].round(3))
print(f"\n{len(off)} session-cameras with |peak lag| > 0.15 s")
off

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, cam in zip(axes[:2], CAMERAS):
    sel = lick_records[lick_records.camera == cam]
    _, arr = st.group_means(sel["trace"], sel["subject"])
    pu.plot_mean_sem(ax, LICK_LAGS, arr, C_ALL, "animals", show_rows=True, label_n=True)
    ax.axvline(0, color=PALETTE["neutral"], ls="--", lw=0.8)
    ax.set(title=f"{CAM_LABEL[cam]}: lick-triggered ME", xlabel="Time from lick (s)", ylabel=f"ME ({ME_NORM})")
    style_ax(ax)
ax = axes[2]
for cam, c in zip(CAMERAS, (OKABE_ITO["blue"], OKABE_ITO["vermillion"])):
    ax.hist(clock.loc[clock.camera == cam, "peak_lag_s"], bins=np.arange(-0.3, 0.31, 0.02),
            color=c, alpha=0.6, label=CAM_LABEL[cam])
ax.set(xlabel="Peak lag per session (s)", ylabel="Sessions", title="Lick-triggered peak lag")
ax.legend()
style_ax(ax)
fig.tight_layout()
save_fig(fig, "men00_clock_check", fig_dir=FIG_DIR, save=SAVE_FIG)

### Per-trial alignment helper

`trial_records` builds one session-mean trace per session × camera × condition, aligned to a trial
column, from boolean trial masks. Go-cue windows are cut at the next go cue so the next trial
does not leak in.

In [ ]:
def trial_records(conds, align_col, lags, censor=True, min_trials=3):
    # conds: {name: function(trials) -> boolean mask}
    out = []
    for s in sessions:
        tr = s["trials"]
        for cam in CAMERAS:
            x = s["me_n"][cam]
            if x is None:
                continue
            for name, fn in conds.items():
                m = fn(tr).fillna(False).to_numpy(bool) & tr[align_col].notna().to_numpy()
                if m.sum() < min_trials:
                    continue
                seg = su.peri_event_grid(x, s["t0"], FS, tr.loc[m, align_col], lags,
                                    censor_after=tr.loc[m, "next_cue"] if censor else None)
                out.append(dict(subject=s["subject"], session=s["session"], camera=cam,
                                cond=name, n=int(m.sum()), trace=np.nanmean(seg, axis=0)))
    return pd.DataFrame(out)


def trial_scalars(windows, align_col):
    # One row per trial and camera: mean ME in each window relative to align_col.
    out = []
    for s in sessions:
        tr = s["trials"]
        for cam in CAMERAS:
            x = s["me_n"][cam]
            if x is None:
                continue
            d = tr.copy()
            d["subject"], d["session"], d["camera"] = s["subject"], s["session"], cam
            for name, (a, b, col) in windows.items():
                d[name] = su.window_mean_grid(x, s["t0"], FS, tr[col], a, b)
            out.append(d)
    return pd.concat(out, ignore_index=True)


def panel_traces(ax, records, lags, conds, cam, xlabel, show_animals=False):
    for name, color in conds.items():
        sel = records[(records.camera == cam) & (records.cond == name)]
        if sel.empty:
            continue
        _, arr = st.group_means(sel["trace"], sel["subject"])
        pu.plot_mean_sem(ax, lags, arr, color, name, show_rows=show_animals, label_n=True)
    ax.axvline(0, color=PALETTE["neutral"], ls="--", lw=0.8)
    ax.set(xlabel=xlabel, ylabel=f"ME ({ME_NORM})", title=CAM_LABEL[cam])
    ax.legend(fontsize=8)
    style_ax(ax)


def paired_animals(ax, df, col, conds, colors, ylabel):
    # Per-animal means of a scalar for each condition, joined by a line per animal.
    sess = df.groupby(["subject", "session", "cond"])[col].mean().reset_index()
    anim = sess.groupby(["subject", "cond"])[col].mean().unstack("cond").reindex(columns=conds)
    xs = np.arange(len(conds))
    for _, r in anim.iterrows():
        ax.plot(xs, r.values, color="#bbbbbb", lw=0.8, zorder=1)
    for i, (c, color) in enumerate(zip(conds, colors)):
        ax.scatter(np.full(len(anim), i), anim[c], color=color, s=18, zorder=2)
        ax.errorbar(i + 0.15, anim[c].mean(), anim[c].sem(), color="k", fmt="o", ms=4, capsize=2)
    ax.set_xticks(xs, conds)
    ax.set(ylabel=ylabel, xlim=(-0.5, len(conds) - 0.5))
    style_ax(ax)
    return anim

## 1. Trial-averaged ME

All trials aligned to the go cue, and responded trials aligned to the choice (the first lick after
the go cue, which is also when the outcome is delivered).

In [ ]:
rec_cue_all = trial_records({"all trials": lambda t: t["goCue_start_time_in_session"].notna()},
                            "goCue_start_time_in_session", CUE_LAGS)
rec_choice_all = trial_records({"responded": lambda t: t["responded"]},
                               "choice_time_in_session", CHOICE_LAGS)

fig, axes = plt.subplots(2, 2, figsize=(10, 6.5), sharey="row")
for j, cam in enumerate(CAMERAS):
    panel_traces(axes[0, j], rec_cue_all, CUE_LAGS, {"all trials": C_ALL}, cam,
                 "Time from go cue (s)", show_animals=True)
    panel_traces(axes[1, j], rec_choice_all, CHOICE_LAGS, {"responded": C_ALL}, cam,
                 "Time from choice (s)", show_animals=True)
fig.tight_layout()
save_fig(fig, "men00_1_trial_average", fig_dir=FIG_DIR, save=SAVE_FIG)

## 2. Rewarded vs unrewarded trials

Responded trials split by `earned_reward`. Trials with auto water (`extra_reward`) are left out:
water arrives there without a rewarded choice.

In [ ]:
REW_CONDS = {
    "rewarded": lambda t: t["responded"] & t["rewarded"] & ~t["auto_water"],
    "unrewarded": lambda t: t["responded"] & ~t["rewarded"] & ~t["auto_water"],
}
REW_COLORS = {"rewarded": C_REW, "unrewarded": C_UNREW}
rec_cue_rew = trial_records(REW_CONDS, "goCue_start_time_in_session", CUE_LAGS)
rec_choice_rew = trial_records(REW_CONDS, "choice_time_in_session", CHOICE_LAGS)

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), gridspec_kw=dict(width_ratios=[1, 1, 0.55]))
for j, cam in enumerate(CAMERAS):
    panel_traces(axes[0, j], rec_cue_rew, CUE_LAGS, REW_COLORS, cam, "Time from go cue (s)")
    panel_traces(axes[1, j], rec_choice_rew, CHOICE_LAGS, REW_COLORS, cam, "Time from choice (s)")

sc = trial_scalars({"outcome_me": (*OUTCOME_WIN, "choice_time_in_session")}, "choice_time_in_session")
sc = sc[sc["responded"] & ~sc["auto_water"]].assign(cond=lambda d: np.where(d["rewarded"], "rewarded", "unrewarded"))
rew_anim = {}
for j, cam in enumerate(CAMERAS):
    rew_anim[cam] = paired_animals(axes[j, 2], sc[sc.camera == cam], "outcome_me",
                                   ["rewarded", "unrewarded"], [C_REW, C_UNREW],
                                   f"ME {OUTCOME_WIN[0]}–{OUTCOME_WIN[1]} s after choice")
    axes[j, 2].set_title(CAM_LABEL[cam])
fig.tight_layout()
save_fig(fig, "men00_2_reward", fig_dir=FIG_DIR, save=SAVE_FIG)

## 3. Responded vs no-response trials

No-response trials cluster where the animal disengages, often late in the session, so the
"isolated" set keeps only no-response trials whose neighbours on both sides were responded: those
come from engaged stretches.

In [ ]:
def isolated_ignore(t):
    resp = t["responded"]
    return ~resp & resp.shift(1, fill_value=False) & resp.shift(-1, fill_value=False)

RESP_CONDS = {
    "responded": lambda t: t["responded"],
    "no response": lambda t: ~t["responded"],
    "no response, isolated": isolated_ignore,
}
RESP_COLORS = {"responded": C_RESP, "no response": C_IGN, "no response, isolated": C_IGN_ISO}
rec_cue_resp = trial_records(RESP_CONDS, "goCue_start_time_in_session", CUE_LAGS, min_trials=2)

sc = trial_scalars({"cue_me": (*CUE_WIN, "goCue_start_time_in_session"),
                    "base_me": (*BASELINE_WIN, "goCue_start_time_in_session")},
                   "goCue_start_time_in_session")
sc["cond"] = np.where(sc["responded"], "responded", "no response")

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), gridspec_kw=dict(width_ratios=[1, 1, 0.55]))
for j, cam in enumerate(CAMERAS):
    panel_traces(axes[0, j], rec_cue_resp, CUE_LAGS, RESP_COLORS, cam, "Time from go cue (s)")
resp_anim = {}
for j, cam in enumerate(CAMERAS):
    resp_anim[cam] = paired_animals(axes[1, j], sc[sc.camera == cam], "base_me",
                                    ["responded", "no response"], [C_RESP, C_IGN],
                                    f"ME {BASELINE_WIN[0]}–{BASELINE_WIN[1]} s before go cue")
    axes[1, j].set_title(f"{CAM_LABEL[cam]}: pre-cue baseline")
counts = sc[sc.camera == CAMERAS[0]].groupby("subject")["responded"].agg(["size", "mean"])
ax = axes[0, 2]
ax.bar(range(len(counts)), 1 - counts["mean"], color=C_IGN)
ax.set_xticks(range(len(counts)), counts.index, rotation=60, fontsize=7)
ax.set(ylabel="Fraction no-response trials", title="No-response rate")
style_ax(ax)
axes[1, 2].axis("off")
fig.tight_layout()
save_fig(fig, "men00_3_responded", fig_dir=FIG_DIR, save=SAVE_FIG)

## 4. Instructed and uninstructed lick bouts

Licks from both spouts are grouped into bouts by the library's `annotate_lick_bouts` (a gap over
0.7 s starts a new bout; `bu.lick_bouts`), the grouping `eph_07` also uses. A bout is
**instructed** when it starts 0–2 s after a go cue and **uninstructed** when it starts more than
2 s after the latest go cue and before the next one (`bu.classify_bout_times`). Bouts before the first and after the
last go cue are left out. ME is aligned to bout onset (the first lick of the bout).

Instructed bouts include the reward-consumption licking on rewarded trials and so tend to be
longer; panel c compares the two at matched bout lengths.

In [ ]:
LEN_BINS = [(1, 1), (2, 3), (4, 7), (8, 1000)]
LEN_LABELS = ["1", "2–3", "4–7", "8+"]

bout_tables, bout_records = [], []
for s in sessions:
    cues = s["trials"]["goCue_start_time_in_session"].dropna().to_numpy()
    b = bu.lick_bouts(s["lick_events"])
    instr, uninstr = bu.classify_bout_times(b["onset"], cues, go_response_window_s=INSTRUCTED_S,
                                            iti_min_post_cue_s=INSTRUCTED_S, iti_min_pre_next_s=0.0)
    b["label"] = np.select([b["onset"].isin(instr), b["onset"].isin(uninstr)],
                           ["instructed", "uninstructed"], "other")
    b = b[b["label"] != "other"].copy()
    b["subject"], b["session"] = s["subject"], s["session"]
    b["len_bin"] = pd.cut(b["n_licks"], [0, 1, 3, 7, 1000], labels=LEN_LABELS)
    for cam in CAMERAS:
        x = s["me_n"][cam]
        if x is None:
            continue
        b[f"onset_me_{cam}"] = su.window_mean_grid(x, s["t0"], FS, b["onset"], 0.0, 0.5)
        b[f"pre_me_{cam}"] = su.window_mean_grid(x, s["t0"], FS, b["onset"], -1.0, -0.2)
        for lab in ("instructed", "uninstructed"):
            on = b.loc[b["label"] == lab, "onset"]
            if len(on) < 3:
                continue
            seg = su.peri_event_grid(x, s["t0"], FS, on, BOUT_LAGS)
            bout_records.append(dict(subject=s["subject"], session=s["session"], camera=cam,
                                     cond=lab, n=len(on), trace=np.nanmean(seg, axis=0)))
    bout_tables.append(b)
bouts = pd.concat(bout_tables, ignore_index=True)
bout_records = pd.DataFrame(bout_records)

per_animal = (bouts.groupby(["subject", "session", "label"]).size().unstack("label")
              .groupby("subject").mean().round(1))
print("Bouts per session (mean per animal):")
per_animal

In [ ]:
BOUT_COLORS = {"instructed": C_INSTR, "uninstructed": C_UNINSTR}
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), gridspec_kw=dict(width_ratios=[1, 1, 0.8]))
for j, cam in enumerate(CAMERAS):
    panel_traces(axes[0, j], bout_records, BOUT_LAGS, BOUT_COLORS, cam, "Time from bout onset (s)")

# b: bout length
ax = axes[0, 2]
for lab, c in BOUT_COLORS.items():
    frac = (bouts[bouts.label == lab].groupby(["subject", "len_bin"], observed=False).size()
            .groupby("subject").transform(lambda v: v / v.sum()).unstack("len_bin"))
    ax.errorbar(np.arange(len(LEN_LABELS)) + (0.08 if lab == "uninstructed" else -0.08),
                frac.mean(), frac.sem(), color=c, marker="o", ms=4, capsize=2, label=lab)
ax.set_xticks(range(len(LEN_LABELS)), LEN_LABELS)
ax.set(xlabel="Licks in bout", ylabel="Fraction of bouts (mean over animals)", title="Bout length")
ax.legend(fontsize=8)
style_ax(ax)

# c: onset ME by bout length
for j, cam in enumerate(CAMERAS):
    ax = axes[1, j]
    col = f"onset_me_{cam}"
    for lab, c in BOUT_COLORS.items():
        a = (bouts[bouts.label == lab].groupby(["subject", "session", "len_bin"], observed=False)[col].mean()
             .groupby(["subject", "len_bin"], observed=False).mean().unstack("len_bin"))
        ax.errorbar(np.arange(len(LEN_LABELS)) + (0.08 if lab == "uninstructed" else -0.08),
                    a.mean(), a.sem(), color=c, marker="o", ms=4, capsize=2, label=lab)
    ax.set_xticks(range(len(LEN_LABELS)), LEN_LABELS)
    ax.set(xlabel="Licks in bout", ylabel="ME 0–0.5 s after onset",
           title=f"{CAM_LABEL[cam]}: onset ME by bout length")
    ax.legend(fontsize=8)
    style_ax(ax)

# d: pre-onset ME
ax = axes[1, 2]
pre = bouts.melt(id_vars=["subject", "session", "label"],
                 value_vars=[f"pre_me_{c}" for c in CAMERAS], var_name="camera", value_name="pre_me")
pre_anim = pre.groupby(["subject", "camera", "label"])["pre_me"].mean().unstack("label")
for k, cam in enumerate(CAMERAS):
    d = pre_anim.xs(f"pre_me_{cam}", level="camera")
    for i, (lab, c) in enumerate(BOUT_COLORS.items()):
        ax.scatter(np.full(len(d), 2 * k + i), d[lab], color=c, s=14)
    for _, r in d.iterrows():
        ax.plot([2 * k, 2 * k + 1], r[list(BOUT_COLORS)].values, color="#bbbbbb", lw=0.7, zorder=0)
ax.set_xticks([0.5, 2.5], [CAM_LABEL[c] for c in CAMERAS])
ax.set(ylabel="ME −1.0 to −0.2 s before onset", title="Before the bout")
for lab, c in BOUT_COLORS.items():
    ax.scatter([], [], color=c, s=14, label=lab)
ax.legend(fontsize=8)
style_ax(ax)
fig.tight_layout()
save_fig(fig, "men00_4_bouts", fig_dir=FIG_DIR, save=SAVE_FIG)

## 5. ME events with and without licking

ME events are upward crossings of 2.5 SD that stay above for 30 ms, one per 0.5 s
(`fu.ME_ONSET_KW`, the movement-onset rule of the `fip_*` notebooks), detected separately in each
camera. An event **has licking** when a lick falls within `LICK_WIN` of its onset. Chance is the
same fraction after circularly shifting each session's lick train by at least 30 s
(`su.coincidence_null`), which keeps lick rate and bout structure. Events without licking are
given the task context the `fip_*` notebooks use for transients (`bu.label_context`): within 1.5 s
after a rewarded or unrewarded outcome, within 1.5 s after the cue of an ignored trial, a lick
within ±1 s, or none of these ("quiet").

The converse is also counted: the fraction of lick bouts (any label) with an ME event within
±0.5 s of bout onset.

In [ ]:
CONTEXTS = ["rewarded", "unrewarded", "cue, no response", "licking", "quiet"]  # bu.label_context


ev_rows, ev_records, lick_psth = [], [], []
for s in sessions:
    cues = s["trials"]["goCue_start_time_in_session"].dropna().to_numpy()
    span = (cues.min(), cues.max() + 10.0)
    le = s["lick_events"]
    bouts_s = bu.lick_bouts(le[(le.timestamps >= span[0]) & (le.timestamps < span[1])])
    for cam in CAMERAS:
        x = s["me_n"][cam]
        if x is None:
            continue
        if ME_NORM == "z":
            xz = x
        else:
            xz = mu.normalise(x, "z")  # events are always detected on the z-scored trace
        ev = su.threshold_onsets(s["t0"] + np.arange(len(xz)) / FS, xz, already_z=True, **ME_ONSET_KW)
        ev = ev[(ev >= span[0]) & (ev < span[1])]
        has_lick = su.near_any(ev, s["licks"], *LICK_WIN)
        null = su.coincidence_null(ev, s["licks"], *LICK_WIN, span=span, n_shift=N_SHIFT)
        bout_hit = su.near_any(bouts_s["onset"].to_numpy(), ev, -0.5, 0.5)
        bout_null = su.coincidence_null(bouts_s["onset"].to_numpy(), ev, -0.5, 0.5, span=span, n_shift=N_SHIFT)
        ctx = bu.label_context(ev[~has_lick], s["trials"], s["licks"])
        ev_rows.append(dict(
            subject=s["subject"], session=s["session"], camera=cam,
            minutes=(span[1] - span[0]) / 60, n_events=len(ev),
            frac_with_lick=has_lick.mean() if len(ev) else np.nan, frac_with_lick_null=np.nanmean(null),
            frac_bouts_with_event=bout_hit.mean() if len(bout_hit) else np.nan,
            frac_bouts_with_event_null=np.nanmean(bout_null),
            **{f"nolick: {c}": np.mean(ctx == c) if len(ctx) else np.nan for c in CONTEXTS},
        ))
        for name, m in (("with licking", has_lick), ("without licking", ~has_lick)):
            if m.sum() < 3:
                continue
            ev_records.append(dict(subject=s["subject"], session=s["session"], camera=cam, cond=name,
                                   n=int(m.sum()),
                                   trace=np.nanmean(su.peri_event_grid(x, s["t0"], FS, ev[m], EVENT_LAGS), axis=0)))
            # Lick rate around the events (Hz), 50 ms bins
            edges = np.arange(EVENT_LAGS[0], EVENT_LAGS[-1] + 0.05, 0.05)
            rel = (s["licks"][None, :] - ev[m][:, None]).ravel()
            h = np.histogram(rel, edges)[0] / m.sum() / 0.05
            lick_psth.append(dict(subject=s["subject"], session=s["session"], camera=cam, cond=name, trace=h))
me_events = pd.DataFrame(ev_rows)
ev_records = pd.DataFrame(ev_records)
lick_psth = pd.DataFrame(lick_psth)
LICK_EDGES_MID = np.arange(EVENT_LAGS[0], EVENT_LAGS[-1], 0.05) + 0.025

ev_anim = me_events.groupby(["camera", "subject"]).mean(numeric_only=True)
ev_anim["events_per_min"] = (me_events.assign(r=me_events.n_events / me_events.minutes)
                             .groupby(["camera", "subject"])["r"].mean())
ev_anim.groupby("camera").agg(["mean", "sem"]).T.round(3)

In [ ]:
EV_COLORS = {"with licking": C_LICK, "without licking": C_NOLICK}
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), gridspec_kw=dict(width_ratios=[1, 1, 0.8]))
for j, cam in enumerate(CAMERAS):
    panel_traces(axes[0, j], ev_records, EVENT_LAGS, EV_COLORS, cam, "Time from ME event onset (s)")
    ax = axes[1, j]
    for name, c in EV_COLORS.items():
        sel = lick_psth[(lick_psth.camera == cam) & (lick_psth.cond == name)]
        _, arr = st.group_means(sel["trace"], sel["subject"])
        pu.plot_mean_sem(ax, LICK_EDGES_MID, arr, c, name, label_n=True)
    ax.axvline(0, color=PALETTE["neutral"], ls="--", lw=0.8)
    ax.axvspan(*LICK_WIN, color="#eeeeee", zorder=0, lw=0)  # the window that defines the two classes
    ax.set(xlabel="Time from ME event onset (s)", ylabel="Lick rate (Hz)", title=f"{CAM_LABEL[cam]}: licks around events")
    ax.legend(fontsize=8)
    style_ax(ax)

ax = axes[0, 2]
for k, cam in enumerate(CAMERAS):
    d = ev_anim.loc[cam]
    for i, (col, c) in enumerate((("frac_with_lick", C_LICK), ("frac_with_lick_null", "#bbbbbb"))):
        ax.scatter(np.full(len(d), 2 * k + i), d[col], color=c, s=14, zorder=2)
    for _, r in d.iterrows():
        ax.plot([2 * k, 2 * k + 1], [r.frac_with_lick, r.frac_with_lick_null], color="#cccccc", lw=0.7, zorder=1)
ax.set_xticks([0, 1, 2, 3], ["bottom", "chance", "side", "chance"])
ax.set(ylabel="Fraction of ME events with a lick", ylim=(0, 1), title="Events with licking")
style_ax(ax)

ax = axes[1, 2]
ctx_cols = [f"nolick: {c}" for c in CONTEXTS]
ctx_colors = [C_REW, C_UNREW, C_IGN, C_LICK, C_NOLICK]
w = 0.16
for k, cam in enumerate(CAMERAS):
    d = ev_anim.loc[cam, ctx_cols]
    for i, (col, c) in enumerate(zip(ctx_cols, ctx_colors)):
        ax.bar(k + (i - 2) * w, d[col].mean(), w, yerr=d[col].sem(), color=c,
               label=col.replace("nolick: ", "") if k == 0 else None, capsize=2)
ax.set_xticks([0, 1], [CAM_LABEL[c] for c in CAMERAS])
ax.set(ylabel="Fraction of events without licking", title="Where events without licking fall")
ax.legend(fontsize=7)
style_ax(ax)
fig.tight_layout()
save_fig(fig, "men00_5_me_events", fig_dir=FIG_DIR, save=SAVE_FIG)

## 6. ME across the session

Running mean of ME over `ROLL_WIN_S` s, every `ROLL_STEP_S` s, against minutes from the first go
cue, with lick rate and the fraction of responded trials over the same windows. Sessions are
averaged within animal where at least `MIN_SESSIONS_AT_T` of that animal's sessions reach that
time; animals are then averaged where at least half of them contribute.

In [ ]:
def per_animal_timecourse(series_by_session, t_grid):
    # series_by_session: list of (subject, t, y). Returns (n_animals, len(t_grid)) array.
    rows = []
    for subj in SUBJECTS:
        ys = [np.interp(t_grid, t, y, left=np.nan, right=np.nan)
              for s_, t, y in series_by_session if s_ == subj]
        if not ys:
            continue
        ys = np.stack(ys)
        m = np.nanmean(ys, axis=0)
        m[np.sum(np.isfinite(ys), axis=0) < min(MIN_SESSIONS_AT_T, len(ys))] = np.nan
        rows.append(m)
    arr = np.stack(rows)
    arr[:, np.sum(np.isfinite(arr), axis=0) < max(1, arr.shape[0] / 2)] = np.nan
    return arr


T_GRID = np.arange(0, 120 * 60, ROLL_STEP_S)  # s from the first go cue
roll = {cam: [] for cam in CAMERAS}
lick_rate, resp_rate = [], []
for s in sessions:
    for cam in CAMERAS:
        x = s["me_n"][cam]
        if x is None:
            continue
        t_off, m = su.rolling_mean(x, FS, ROLL_WIN_S, ROLL_STEP_S)
        roll[cam].append((s["subject"], s["t0"] + t_off, m))
    t_end = s["trials"]["goCue_start_time_in_session"].max()
    tc = np.arange(0, t_end, ROLL_STEP_S)
    lr = np.array([np.sum(np.abs(s["licks"] - t) < ROLL_WIN_S / 2) for t in tc]) / ROLL_WIN_S
    lick_rate.append((s["subject"], tc, lr))
    tr = s["trials"]
    rr = np.array([tr.loc[(tr.goCue_start_time_in_session - t).abs() < ROLL_WIN_S / 2, "responded"].mean()
                   for t in tc])
    resp_rate.append((s["subject"], tc, rr))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, cam in zip(axes[:2], CAMERAS):
    arr = per_animal_timecourse(roll[cam], T_GRID)
    pu.plot_mean_sem(ax, T_GRID / 60, arr, C_ALL, "animals", show_rows=True, label_n=True)
    ax.set(xlabel="Minutes from first go cue", ylabel=f"ME ({ME_NORM}), {ROLL_WIN_S:.0f}-s mean",
           title=CAM_LABEL[cam])
    style_ax(ax)
ax = axes[2]
pu.plot_mean_sem(ax, T_GRID / 60, per_animal_timecourse(lick_rate, T_GRID), OKABE_ITO["bluish_green"], "lick rate (Hz)", label_n=True)
ax.set(xlabel="Minutes from first go cue", ylabel="Licks / s", title="Lick rate and responding")
ax2 = ax.twinx()
pu.plot_mean_sem(ax2, T_GRID / 60, per_animal_timecourse(resp_rate, T_GRID), C_IGN_ISO, "responded fraction", label_n=True)
ax2.set_ylabel("Fraction responded", color=C_IGN_ISO)
ax2.set_ylim(0, 1.05)
ax.legend(loc="lower left", fontsize=8)
ax2.legend(loc="lower right", fontsize=8)
style_ax(ax)
fig.tight_layout()
save_fig(fig, "men00_6_session_timecourse", fig_dir=FIG_DIR, save=SAVE_FIG)

## 7. ME by recent reward history

Trials are binned by the run of consecutive rewards (positive) or failures (negative) *entering*
the trial, i.e. `num_reward_past` of the previous trial (Rachel's definition; no-response trials
count as failures). Only responded trials are averaged, so the current trial's own response is
held fixed. Rows: go-cue-aligned ME per bin; the pre-cue baseline by bin; ME in the first second after
the cue by bin (grey lines are animals).

In [ ]:
STREAK_BINS = [(lab, fn, c) for lab, fn, c in fu.STREAK_BINS]   # fail>=3 ... rew>=3
STREAK_LABELS = [b[0] for b in STREAK_BINS]
hist_conds = {lab: (lambda t, fn=fn: t["responded"] & t["streak_before"].map(lambda v: bool(fn(v)) if pd.notna(v) else False))
              for lab, fn, _ in STREAK_BINS}
HIST_COLORS = {lab: c for lab, _, c in STREAK_BINS}
rec_hist = trial_records(hist_conds, "goCue_start_time_in_session", CUE_LAGS, min_trials=3)

sc = trial_scalars({"base_me": (*BASELINE_WIN, "goCue_start_time_in_session"),
                    "cue_me": (*CUE_WIN, "goCue_start_time_in_session")},
                   "goCue_start_time_in_session")
sc = sc[sc["responded"] & sc["streak_before"].notna()].copy()
sc["cond"] = None
for lab, fn, _ in STREAK_BINS:
    sc.loc[sc["streak_before"].map(lambda v: bool(fn(v))), "cond"] = lab
sc = sc[sc["cond"].notna()]

fig, axes = plt.subplots(3, 2, figsize=(11, 10), gridspec_kw=dict(height_ratios=[1.3, 1, 1]))
hist_anim = {}
for j, cam in enumerate(CAMERAS):
    panel_traces(axes[0, j], rec_hist, CUE_LAGS, HIST_COLORS, cam, "Time from go cue (s)")
    hist_anim[cam] = {}
    for i, (col, lab) in enumerate((("base_me", f"Pre-cue baseline {BASELINE_WIN} s"),
                                     ("cue_me", f"After the cue {CUE_WIN} s"))):
        ax = axes[1 + i, j]
        a = (sc[sc.camera == cam].groupby(["subject", "session", "cond"])[col].mean()
             .groupby(["subject", "cond"]).mean().unstack("cond").reindex(columns=STREAK_LABELS))
        hist_anim[cam][col] = a
        for _, r in a.iterrows():
            ax.plot(range(len(STREAK_LABELS)), r.values, color="#cccccc", lw=0.7, zorder=0)
        ax.errorbar(range(len(STREAK_LABELS)), a.mean(), a.sem(), marker="o", capsize=2, color=C_ALL)
        ax.set_xticks(range(len(STREAK_LABELS)), STREAK_LABELS)
        ax.set(xlabel="Reward history entering the trial", ylabel=f"ME ({ME_NORM})",
               title=f"{CAM_LABEL[cam]}: {lab}")
        style_ax(ax)
fig.tight_layout()
save_fig(fig, "men00_7_reward_history", fig_dir=FIG_DIR, save=SAVE_FIG)

n_hist = sc[sc.camera == CAMERAS[0]].groupby(["subject", "cond"]).size().unstack("cond").reindex(columns=STREAK_LABELS)
print("Responded trials per history bin, per animal:")
n_hist

## 8. Numbers

Per camera, mean ± SEM across animals, with a Wilcoxon signed-rank test across animals for the
paired comparisons.

In [ ]:
def paired(a, b):
    d = (a - b).dropna()
    m, p, n = st.wilcoxon_animals(d.values)
    return f"{m:+.3f} ± {d.sem():.3f} (p={p:.3g}, n={n})"

summary = []
for cam in CAMERAS:
    ra = rew_anim[cam]
    summary.append((cam, f"rewarded − unrewarded, ME {OUTCOME_WIN} s after choice", paired(ra["rewarded"], ra["unrewarded"])))
    pa = resp_anim[cam]
    summary.append((cam, f"responded − no response, pre-cue baseline {BASELINE_WIN}", paired(pa["responded"], pa["no response"])))
    pre = pre_anim.xs(f"pre_me_{cam}", level="camera")
    summary.append((cam, "instructed − uninstructed bouts, ME −1.0 to −0.2 s before onset", paired(pre["instructed"], pre["uninstructed"])))
    ea = ev_anim.loc[cam]
    summary.append((cam, "ME events per min", f"{ea.events_per_min.mean():.2f} ± {ea.events_per_min.sem():.2f}"))
    summary.append((cam, f"fraction of ME events with a lick in {LICK_WIN} s − chance",
                    paired(ea["frac_with_lick"], ea["frac_with_lick_null"])))
    summary.append((cam, "fraction of lick bouts with an ME event within ±0.5 s − chance",
                    paired(ea["frac_bouts_with_event"], ea["frac_bouts_with_event_null"])))
    h = hist_anim[cam]["base_me"]
    summary.append((cam, "pre-cue baseline, rew>=3 − fail>=3", paired(h["rew>=3"], h["fail>=3"])))
    h = hist_anim[cam]["cue_me"]
    summary.append((cam, "0–1 s after cue, rew>=3 − fail>=3", paired(h["rew>=3"], h["fail>=3"])))
summary = pd.DataFrame(summary, columns=["camera", "quantity", "mean ± SEM across animals"])
summary